# 06 – Deploy da Calculadora de LTV

**Fase CRISP-DM:** Deployment

Nesta etapa, o modelo final de **Regressão Linear** é convertido em uma calculadora de LTV em Excel, com inputs editáveis, fórmula nativa do Excel, documentação e validação contra o cálculo em Python.

In [ ]:
from pathlib import Path
import sys
import warnings

import pandas as pd
from IPython.display import display

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "config" / "config.yaml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))

from src.deploy import (
    create_ltv_calculator_workbook,
    default_test_clients,
    fit_linear_calculator_artifacts,
    validate_excel_against_python,
)

df_final = pd.read_csv(PROJECT_ROOT / "data" / "final" / "ltv_base_final.csv")
display(df_final.head())

## Parâmetros do modelo final

In [ ]:
artifacts = fit_linear_calculator_artifacts(df_final)

model_summary = pd.DataFrame(
    {
        "métrica": ["Modelo", "R2", "RMSE", "MAE", "Intercepto", "Nº de features transformadas"],
        "valor": [
            "Regressão Linear",
            artifacts.metrics["R2"],
            artifacts.metrics["RMSE"],
            artifacts.metrics["MAE"],
            artifacts.intercept,
            len(artifacts.feature_names),
        ],
    }
)
display(model_summary)

coef_table = pd.DataFrame(
    {
        "feature_transformada": artifacts.feature_names,
        "coeficiente": artifacts.coefficients,
    }
)
display(coef_table)

## Geração do arquivo Excel

In [ ]:
output_path = PROJECT_ROOT / "models" / "ltv_calculadora_linear.xlsx"
workbook_path = create_ltv_calculator_workbook(artifacts, output_path)
print(f"Arquivo gerado em: {workbook_path}")

## Validação Python vs Excel

In [ ]:
validation_df = validate_excel_against_python(
    workbook_path=workbook_path,
    artifacts=artifacts,
    test_clients=default_test_clients(),
)
display(validation_df)